# Resumen Sinteticos (Observacional): Beta=0 vs Beta optima, agregado sobre los 11 datasets

A diferencia de `Resumen_Observacional.ipynb` (Experimentos 1 y 2, donde el grafo es siempre el mismo
y lo que varia es el ruido), este notebook resume el **Experimento 3**: aqui el ruido se fija
(Gaussiano o Gamma) y lo que varia son los 11 grafos/datasets sinteticos del benchmark
(`3-chain-linear`, ..., `triangle-non-linear`). Se compara el modelo base (`Beta=0`, sin termino HSIC)
frente al modelo con la `Beta` optima elegida segun un criterio multi-metrica sobre `MMD` y `RF Acc`
(las dos metricas observacionales que se registran en Experimento 3), agregando sobre los 11 datasets
x 10 semillas, y se comprueba con un test de Wilcoxon pareado (por `Dataset` y `Seed`) si la mejora es
significativa.

Los resultados de ruido **Gaussiano** y **Gamma** se muestran en dos tablas separadas dentro de este
mismo notebook.

Todo el analisis se repite para `N=50` y `N=100`.

> Nota: a la fecha de creacion de este notebook, los CSV fuente en
> `Experimento3 datasets/Obervacional/` todavia se estan generando (los notebooks
> `Datasets_Sinteticos_Observacional[ gamma].ipynb` siguen en ejecucion), asi que puede haber menos de
> los 11 datasets disponibles. Este notebook lo detecta automaticamente y lo indica en la columna
> `Cobertura`/`Nota` de cada tabla; basta con volver a ejecutarlo cuando los datos esten completos.


In [71]:
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"

ALPHA = 0.05
METRICAS = ["MMD", "RF Acc"]  # menor es mejor en las 2 (RF Acc bajo = generador indistinguible del real)


## Fuentes de datos (Gaussiano vs Gamma)

In [72]:
# A diferencia de Resumen_Observacional.ipynb (un CSV por tipo de ruido, mismo grafo), aqui cada CSV
# ya contiene los 11 datasets (columna 'Dataset'); solo hay dos fuentes, una por tipo de ruido.
# El CSV sin sufijo es el Gaussiano (Experimento 3 no usa el sufijo "_gausian" de Experimento 1/2);
# "_gamma" es el unico sufijo que existe.
FUENTES = {
    "Gaussiano": NOTEBOOKS_DIR / "Experimento3 datasets" / "Obervacional" / "tablas" / "datasets_sinteticos_observacional.csv",
    "Gamma": NOTEBOOKS_DIR / "Experimento3 datasets" / "Obervacional" / "tablas" / "datasets_sinteticos_observacional_gamma.csv",
}

for nombre, path in FUENTES.items():
    assert path.exists(), f"No existe: {path}"

# Los 11 datasets del benchmark (misma lista que Datasets_Sinteticos_Observacional[ gamma].ipynb);
# solo se usa para medir cobertura, no para filtrar filas.
TOTAL_DATASETS_ESPERADOS = 11


## Funcion de seleccion de la Beta optima

Misma logica multi-metrica que `Resumen_Observacional.ipynb` / `Resumen_Intervencional.ipynb`: es
generica sobre la lista de metricas y sobre que columnas tenga el DataFrame, asi que no necesita
cambios para trabajar aqui con `MMD`/`RF Acc` en vez de `MAE Z`/`HSIC(Z,X)`/`HSIC(Z,Y)`. La unica
diferencia de fondo es que el `df` que recibe ya contiene los 11 datasets sinteticos mezclados (columna
`Dataset`): `groupby("Beta")` promedia automaticamente sobre Dataset x Seed, que es justo la "media
total sobre todos los grafos" que se busca.


In [73]:
def beta_optima(df: pd.DataFrame, n_filter: int, metrics=METRICAS):
    """Selecciona la Beta optima (Beta != 0) segun un criterio multi-metrica, agregando sobre TODOS
    los datasets (columna 'Dataset') y semillas presentes en df, para un N fijo.

    Beta=0 se usa solo como referencia (baseline) y nunca puede ser el resultado.

    1) Filtra N == n_filter, promedia cada metrica por Beta (sobre los datasets x seeds disponibles) y
       descarta Beta=0 del conjunto de candidatas.
    2) Para cada metrica, halla la Beta > 0 que minimiza su media -> betas candidatas.
    3) Para cada beta candidata, cuenta cuantas metricas mejoran/empeoran frente a Beta=0 y
       calcula la mejora relativa (%) de cada metrica frente al baseline.
    4) Se ordenan las candidatas por, en este orden de prioridad:
         a) menos metricas empeoradas (una candidata que no empeora ninguna gana siempre),
         b) mas metricas mejoradas,
         c) mayor mejora relativa total (suma de mejoras relativas),
         d) ULTIMO desempate, solo si las anteriores empatan exactamente: mayor mejora
            relativa en una unica metrica.
       Se elige la primera de ese orden.

    Devuelve (baseline: Series, beta_opt: float, valores_opt: Series, candidatas: DataFrame,
    criterio: str).
    """
    df_n = df[df["N"] == n_filter]
    media_por_beta = df_n.groupby("Beta")[list(metrics)].mean()

    baseline = media_por_beta.loc[0.0]
    media_no_cero = media_por_beta.drop(index=0.0)

    betas_candidatas = sorted(set(media_no_cero[m].idxmin() for m in metrics))

    filas_candidatas = []
    for beta in betas_candidatas:
        valores = media_no_cero.loc[beta]
        mejora_relativa = (baseline - valores) / baseline
        filas_candidatas.append({
            "Beta": beta,
            "n_mejoradas": int((valores < baseline).sum()),
            "n_empeoradas": int((valores > baseline).sum()),
            "mejora_relativa_total": float(mejora_relativa.sum()),
            "mejora_relativa_max": float(mejora_relativa.max()),
        })

    candidatas = pd.DataFrame(filas_candidatas).sort_values(
        by=["n_empeoradas", "n_mejoradas", "mejora_relativa_total", "mejora_relativa_max"],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)

    ganadora = candidatas.iloc[0]
    beta_opt = float(ganadora["Beta"])
    valores_opt = media_por_beta.loc[beta_opt]

    if ganadora["n_empeoradas"] == 0:
        criterio = "domina al baseline (mejora >=1 metrica sin empeorar ninguna)"
    else:
        criterio = "mejor compromiso (ninguna beta evita empeorar alguna metrica)"

    return baseline, beta_opt, valores_opt, candidatas, criterio


## Funcion de test de Wilcoxon pareado (por Dataset y Seed)

Para cada tipo de ruido y cada N, se comparan los valores en `Beta=0` frente a `Beta=beta_optima`,
emparejados por `(Dataset, Seed)`: cada fila del CSV es un grafo (`Dataset`) entrenado con una semilla
(`Seed`) concreta, asi que el par natural es uno por cada combinacion grafo+semilla (hasta 11 x 10 =
110 pares por N) -- no solo por semilla, como en `Resumen_Observacional.ipynb` (que no tenia dimension
de grafo), ni por `(Seed, Y_do)`, como en `Resumen_Intervencional.ipynb` (que anadia una dimension de
intervencion). Test unidireccional (`alternative='less'`): H0 = no hay diferencia; H1 = el valor con
la beta optima es menor (mejor) que con beta=0.


In [74]:
def wilcoxon_beta(df: pd.DataFrame, beta_opt: float, n_filter: int, metrics=METRICAS):
    """Wilcoxon signed-rank pareado (por Dataset y Seed) entre Beta=0 y Beta=beta_opt, por metrica.

    H0: no hay diferencia. H1 (alternative='less'): el valor en beta_opt es menor (mejor) que en Beta=0.
    Devuelve (p_valores: dict metrica->p, n_pares: int).
    """
    df_n = df[df["N"] == n_filter]
    base = df_n[df_n["Beta"] == 0.0].set_index(["Dataset", "Seed"])[list(metrics)]
    opt = df_n[df_n["Beta"] == beta_opt].set_index(["Dataset", "Seed"])[list(metrics)]
    pares_comunes = sorted(set(base.index) & set(opt.index))
    base = base.loc[pares_comunes]
    opt = opt.loc[pares_comunes]

    p_valores = {}
    for m in metrics:
        try:
            _, p = wilcoxon(opt[m].values, base[m].values, alternative="less")
        except ValueError:
            p = float("nan")
        p_valores[m] = p
    return p_valores, len(pares_comunes)


## Calculo (tabla resumen + p-valores) por N, para cada tipo de ruido


In [75]:
def analizar_ruido(path: Path, n_filter: int, metrics=METRICAS):
    """Ejecuta beta_optima + wilcoxon_beta sobre TODOS los datasets presentes en `path`, a un N fijo.

    Devuelve (fila_resumen: dict, candidatas: DataFrame).
    """
    df = pd.read_csv(path)
    df_n = df[df["N"] == n_filter]
    n_datasets_presentes = df_n["Dataset"].nunique()
    cobertura = f"{n_datasets_presentes}/{TOTAL_DATASETS_ESPERADOS}"
    nota = "" if n_datasets_presentes >= TOTAL_DATASETS_ESPERADOS else f"\u26a0 datos parciales ({cobertura})"

    baseline, beta_opt, valores_opt, candidatas, criterio = beta_optima(df, n_filter=n_filter, metrics=metrics)
    p_valores, n_pares = wilcoxon_beta(df, beta_opt, n_filter=n_filter, metrics=metrics)

    fila = {
        "N": n_filter,
        "beta_optima": beta_opt,
        "MMD beta=0": baseline["MMD"],
        "MMD beta_optima": valores_opt["MMD"],
        "MMD p_valor": p_valores["MMD"],
        "RF Acc beta=0": baseline["RF Acc"],
        "RF Acc beta_optima": valores_opt["RF Acc"],
        "RF Acc p_valor": p_valores["RF Acc"],
        "n_pares": n_pares,
        "Cobertura": cobertura,
        "Criterio": criterio,
        "Nota": nota,
    }
    return fila, candidatas


resultados = {}
diagnostico = {}
for nombre_ruido, path in FUENTES.items():
    filas = []
    for n_filter in (50, 100):
        fila, candidatas = analizar_ruido(path, n_filter)
        filas.append(fila)
        diagnostico[(nombre_ruido, n_filter)] = candidatas
    resultados[nombre_ruido] = pd.DataFrame(filas)

resumen_gaussiano = resultados["Gaussiano"]
resumen_gamma = resultados["Gamma"]


## Funciones de formato y resaltado en negrita


In [76]:
COLUMNA_A_METRICA = {
    "MMD beta_optima": "MMD",
    "RF Acc beta_optima": "RF Acc",
}


def formatear(resumen: pd.DataFrame):
    cols_num = [c for c in resumen.columns if c not in ("N", "beta_optima", "Criterio", "Nota", "Cobertura")]
    fmt = resumen.copy()
    fmt[cols_num] = fmt[cols_num].round(5)
    fmt["beta_optima"] = fmt["beta_optima"].round(2)
    return fmt


def tabla_con_negrita(resumen: pd.DataFrame):
    """Tabla resumen con las celdas 'beta_optima' en negrita cuando su p-valor (Wilcoxon) < ALPHA.

    Solo aplica en la visualizacion del notebook (un CSV plano no admite negrita).
    """
    fmt = formatear(resumen)

    def resaltar(row):
        estilos = []
        for col in row.index:
            metrica = COLUMNA_A_METRICA.get(col)
            if metrica is not None and pd.notna(row[f"{metrica} p_valor"]) and row[f"{metrica} p_valor"] < ALPHA:
                estilos.append("font-weight: bold")
            else:
                estilos.append("")
        return estilos

    return fmt.style.apply(resaltar, axis=1)


## Ruido Gaussiano

In [77]:
tabla_con_negrita(resumen_gaussiano)


,N,beta_optima,MMD beta=0,MMD beta_optima,MMD p_valor,RF Acc beta=0,RF Acc beta_optima,RF Acc p_valor,n_pares,Cobertura,Criterio,Nota
0,50,0.300000,0.025120,0.023320,0.000050,0.671360,0.669090,0.413490,110,11/11,domina al baseline (mejora >=1 metrica sin empeorar ninguna),
1,100,0.400000,0.018520,0.017890,0.002010,0.644770,0.631360,0.015000,110,11/11,domina al baseline (mejora >=1 metrica sin empeorar ninguna),


## Ruido Gamma

In [78]:
tabla_con_negrita(resumen_gamma)


,N,beta_optima,MMD beta=0,MMD beta_optima,MMD p_valor,RF Acc beta=0,RF Acc beta_optima,RF Acc p_valor,n_pares,Cobertura,Criterio,Nota
0,50,0.300000,0.047540,0.046120,0.001740,0.689770,0.683410,0.094690,110,11/11,domina al baseline (mejora >=1 metrica sin empeorar ninguna),
1,100,0.100000,0.043790,0.043910,0.119490,0.661360,0.668180,0.944350,110,11/11,mejor compromiso (ninguna beta evita empeorar alguna metrica),


## Comparacion Gaussiano vs Gamma

Beta optima elegida y p-valores por N y tipo de ruido, para ver de un vistazo si las conclusiones
cambian entre Gaussiano y Gamma.


In [79]:
comparacion = pd.concat(
    {nombre: df.set_index("N")[["beta_optima", "MMD p_valor", "RF Acc p_valor", "n_pares", "Cobertura"]]
     for nombre, df in resultados.items()},
    names=["Ruido", "N"]
)
comparacion


beta_optima  MMD p_valor  RF Acc p_valor  n_pares Cobertura
Ruido     N                                                               
Gaussiano 50           0.3     0.000047        0.413494      110     11/11
          100          0.4     0.002011        0.015004      110     11/11
Gamma     50           0.3     0.001742        0.094689      110     11/11
          100          0.1     0.119486        0.944346      110     11/11

## Guardar CSVs resumen


In [80]:
OUT_PATH_GAUSSIANO = NOTEBOOKS_DIR / "tablas" / "resumen_sinteticos_observacional_beta_gaussiano.csv"
OUT_PATH_GAMMA = NOTEBOOKS_DIR / "tablas" / "resumen_sinteticos_observacional_beta_gamma.csv"

resumen_gaussiano.to_csv(OUT_PATH_GAUSSIANO, index=False)
resumen_gamma.to_csv(OUT_PATH_GAMMA, index=False)

print(f"Guardado en: {OUT_PATH_GAUSSIANO}")
print(f"Guardado en: {OUT_PATH_GAMMA}")


Guardado en: /Users/clau/Documents/TFM_NUEVO/CODIGO_COMPLETO/kacgm-hsic/kacgm_hsic/notebooks/tablas/resumen_sinteticos_observacional_beta_gaussiano.csv
Guardado en: /Users/clau/Documents/TFM_NUEVO/CODIGO_COMPLETO/kacgm-hsic/kacgm_hsic/notebooks/tablas/resumen_sinteticos_observacional_beta_gamma.csv
